# Streaming

Streaming reduces the latency between generating data and the user receiving it. There are two types frequently used with Agents:

In [12]:
from langchain.agents import create_agent

agent = create_agent(
    model = "ollama:qwen2.5:3b",
    system_prompt="You are a full-stack comedian"
)

## No Streaming (invoke)

In [13]:
result = agent.invoke({"messages": [{"role": "user", "content": "Tell me a joke"}]})
print(result["messages"][-1].content)

Sure, here's one for you:

Why don't scientists trust atoms?

Because they make up everything! 

It's a classic atomic pun - it plays on the double meaning of "make up," as in compose or to lie. Hope that tickled your funny bone!


# values
You have seen this streaming mode in our examples so far.

In [14]:
#Stream = values
for step in agent.stream(
    {"messages": [{"role": "user", "content": "Tell me a dank joke"}]},
    stream_mode="values"
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me a dank joke
================================== Ai Message ==================================

Sure thing, here’s one that'll leave you wanting more:

Why don't scientists trust atoms?

Because they make up everything!


# Messages
Messages stream data token by token - the lowest latency possible. This is perfect for interactive applications like chatbots.

In [15]:
for token, metadata in agent.stream(
    {"messages": {"role":"user", "content": "Write a family friendly poem."}},
    stream_mode="messages"
):
    print(f"{token.content}", end="")

In the heart of our home, where the sun does peek,
We have rooms for all, with walls that keep.
The living room, oh so cozy and warm,
Where stories we share, in laughter and charm.

There's a kitchen there where the good times start,
With pots and pans, and a little bit of art.
Mom's hands are quick as they stir up a meal,
Every dish brings us close together again.

Upstairs, bedrooms for all to spread,
Each has its own light from above.
Underneath them all, we find our bed,
Our comfort is found in the softest of bed sheets and sheets.

Our backyard, where adventures can play,
With toys that come alive as they sway.
The flowers there bloom into a colorful show,
As bees buzz by with their sweet perfume shows.

In every room, under every sky,
We are home, we are at peace.
For our family's love fills the air so free,
And together we make memories to share.

# Tools can stream too!
Streaming generally means delivering information to the user before the final result is ready. There are many cases where this is useful. A `get_stream_writer writer` allows you to easily stream `custom`data from sources you create.

In [16]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()
    # stream any arbitrary data
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"


agent = create_agent(
    model="ollama:qwen2.5:3b",
    tools=[get_weather],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "custom"],
):
   print(chunk)

('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='895c1571-6074-48e5-aa4c-bc3fc3277419')]})
('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='895c1571-6074-48e5-aa4c-bc3fc3277419'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-07-10T16:51:37.971173Z', 'done': True, 'done_reason': 'stop', 'total_duration': 960734958, 'load_duration': 163555250, 'prompt_eval_count': 152, 'prompt_eval_duration': 48354000, 'eval_count': 20, 'eval_duration': 739492000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019f4cf1-19b1-7503-b571-4c7e60adb9b3-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'SF'}, 'id': '50e55839-5c8e-4dc9-91a4-c833549a5fb3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_tokens': 20, '

In [17]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["custom"],
):
    print(chunk)

('custom', 'Looking up data for city: SF')
('custom', 'Acquired data for city: SF')
